# NexusMart data cleaning milestone

## Purpose and evidence boundary

This notebook documents the final state using the reports already generated by the cleaning workflow. Its code cells only read CSV reports and display summaries; they do not modify raw, processed, or rejected data.

The original `01_data_profiling_cleaning.ipynb` preserves the executed raw profiling and investigation cells. The later cleaning passes are evidenced by `data/processed/`, `data/rejected/`, and the audit logs. The current repository does not contain the original code cells or scripts for every historical date, sentinel, and invalid-item quarantine operation, so this notebook does not claim to rebuild those results.

Do not run `src/finalize_cleaning.py` as part of this report review: it writes the already prepared processed files in place. See the root README for the safe execution notes.

## Observed workflow

1. Profiled the untouched top-level CSVs and recorded the raw baseline and candidate-key/date audits.
2. Removed exact duplicate copies from the processed path, retaining raw inputs; counts are in `duplicate_removal_log.csv`.
3. Investigated missing values, categories, dates, numeric anomalies, and references. Applied documented missing-category/no-discount rules and supported category aliases.
4. Quarantined invalid dates, suspected sentinel records, and invalid order items with reason codes. Kept unresolved numeric anomalies and entity candidates flagged rather than guessing.
5. Finalized the existing processed files, checked types and all 10 reference relationships, and reconciled retained, rejected, and exact-duplicate counts.

The current finalization script covers the final step and reads the prepared processed/rejected directories. It does not reconstruct all prior quarantine operations from the raw datasets.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
REPORT_DIR = PROJECT_ROOT / "reports"

def read_report(name):
    path = REPORT_DIR / name
    if not path.exists():
        raise FileNotFoundError(f"Required report not found: {path}")
    return pd.read_csv(path, low_memory=False)

quality = read_report("consolidated_quality_report.csv")
validation = read_report("cleaning_validation_report.csv")
print("Tables reconciled:", int(quality["row_reconciliation_pass"].sum()), "of", len(quality))
print("Exact duplicate rows retained:", int(quality["exact_duplicates_retained"].sum()))
print("Duplicate primary-key occurrences:", int(quality["unique_key_duplicates"].sum()))
display(quality[["file", "original_rows", "retained_rows", "quarantined_rows",
                 "exact_duplicate_rows_removed", "row_reconciliation_pass"]])
display(validation)

Tables reconciled: 9 of 9
Exact duplicate rows retained: 0
Duplicate primary-key occurrences: 0


,file,original_rows,retained_rows,quarantined_rows,exact_duplicate_rows_removed,row_reconciliation_pass
0,customers.csv,45000,45000,0,0,True
1,locations.csv,650,650,0,0,True
2,orders.csv,163200,158250,1750,3200,True
3,order_items.csv,315143,313250,1893,0,True
4,payments.csv,168111,163586,1229,3296,True
5,products.csv,3200,3200,0,0,True
6,returns.csv,16048,15912,136,0,True
7,sellers.csv,420,416,4,0,True
8,shipping.csv,151228,149122,2106,0,True


,check,status,details
0,row_reconciliation,PASS,9/9 tables reconcile
1,exact_duplicate_rows,PASS,0 exact duplicate rows remain
2,primary_key_duplicates,PASS,0 duplicate primary key occurrences remain
3,numeric_parse,PASS,0 unparseable non-null numeric values
4,referential_integrity,REVIEW,1381 broken and 7937 references to quarantined...
5,reference_classification_reconciliation,PASS,"10 relationships partition into missing, valid..."
6,date_review,REVIEW,1731 invalid or ambiguous customer registratio...
7,reason_coded_rejections,PASS,7118 unique quarantined rows in consolidated r...
8,category_alias_mapping,PASS,0 rows remain under explicitly mapped source l...
9,negative_value_flags,PASS,988 current negative values flagged; statistic...


## Decisions and retained exceptions

The decision log distinguishes completed transformations from unresolved work. Optional and unsupported values remain null; true broken references and references to quarantined parents remain in the exception report for staging disposition.

In [2]:
decisions = read_report("final_cleaning_decision_log.csv")
missing = read_report("final_missing_value_decisions.csv")
display(decisions)
display(missing[missing["missing_values_retained"] > 0])

,category,status,evidence_and_decision
0,Missing values,partially implemented,Documented category/no-discount actions applie...
1,Exact duplicate records,implemented,Existing duplicate-removal counts reconciled w...
2,Duplicate customers/products under different IDs,partially implemented,Candidate report retained; no entity merged wi...
3,Inconsistent categories,implemented,"Existing normalization plus explicit, preserve..."
4,Invalid dates,partially implemented,Audited date columns quarantined; registration...
5,Incorrect data types,partially implemented,Numeric values and nullable IDs converted; dat...
6,Outliers,partially implemented,Known sentinel and invalid quantities quaranti...
7,Broken references,partially implemented,Final references checked against retained and ...


,file,column,documented_action,final_decision,missing_values_retained,business_reason
0,customers.csv,email,Retain missing,Retained as null; unresolved or optional; no i...,1698,Contact detail is optional for analysis; do no...
1,customers.csv,phone,Retain missing,Retained as null; unresolved or optional; no i...,3599,Contact detail is optional for analysis; do no...
3,customers.csv,city_id,Investigate reference; otherwise Unknown,Retained as null; unresolved or optional; no i...,2233,Customer geography is unavailable unless a val...
4,customers.csv,birth_year,Retain missing; flag for demographic analysis,Retained as null; unresolved or optional; no i...,4600,Do not infer age without reliable evidence.
5,locations.csv,province,Investigate; otherwise Unknown,Retained as null; unresolved or optional; no i...,25,Do not infer province from city name alone.
6,locations.csv,latitude,Retain missing unless reliable mapping exists,Retained as null; unresolved or optional; no i...,33,Do not fabricate coordinates.
7,locations.csv,longitude,Retain missing unless reliable mapping exists,Retained as null; unresolved or optional; no i...,31,Do not fabricate coordinates.
8,order_items.csv,discount_amount,Investigate numeric meaning; then resolve,Retained as null; unresolved or optional; no i...,31364,"Missing may mean no discount, but confirm agai..."
10,orders.csv,ship_city_id,Investigate reference; otherwise Unknown,Retained as null; unresolved or optional; no i...,6322,Shipping geography should not be guessed.
11,payments.csv,payment_date,Retain missing; investigate payment status,Retained as null; unresolved or optional; no i...,4924,A missing date does not establish when payment...


## Referential integrity and quarantine evidence

Formatting-only city ID differences are separated from broken keys. A reference to a parent that was quarantined is also kept separate from a truly unmatched key. Missing city fields remain a null-data issue rather than a broken reference.

In [3]:
references = read_report("final_reference_integrity_report.csv")
rejections = read_report("reason_coded_rejection_inventory.csv")
category_map = read_report("category_value_mapping.csv")
display(references)
print("True broken references:", int(references["true_broken_references"].sum()))
print("References to quarantined parents:", int(references["references_to_quarantined_parent"].sum()))
print("Formatting-only matches:", int(references["formatting_mismatch_only"].sum()))
print("Unique quarantined rows:", int(rejections["rows"].sum()))
print("Mapped source labels remaining:", int(category_map["rows_remaining_as_source_label"].sum()))
display(rejections)

,relationship,rows_checked,missing_references,valid_references,references_to_quarantined_parent,formatting_mismatch_only,true_broken_references,handling
0,customers.csv.city_id->locations.csv.city_id,45000,2233,0,0,42767,0,Retained and reported; no unsupported ID mapping
1,sellers.csv.city_id->locations.csv.city_id,416,0,416,0,0,0,Retained and reported; no unsupported ID mapping
2,orders.csv.customer_id->customers.csv.customer_id,158250,0,157932,0,0,318,Retained and reported; no unsupported ID mapping
3,orders.csv.ship_city_id->locations.csv.city_id,158250,6322,0,0,151928,0,Retained and reported; no unsupported ID mapping
4,order_items.csv.order_id->orders.csv.order_id,313250,0,309785,3465,0,0,Retained and reported; no unsupported ID mapping
5,order_items.csv.product_id->products.csv.produ...,313250,0,312325,0,0,925,Retained and reported; no unsupported ID mapping
6,order_items.csv.seller_id->sellers.csv.seller_id,313250,0,312308,942,0,0,Retained and reported; no unsupported ID mapping
7,payments.csv.order_id->orders.csv.order_id,163586,0,161805,1781,0,0,Retained and reported; no unsupported ID mapping
8,shipping.csv.order_id->orders.csv.order_id,149122,0,147477,1645,0,0,Retained and reported; no unsupported ID mapping
9,returns.csv.order_item_id->order_items.csv.ord...,15912,0,15670,104,0,138,Retained and reported; no unsupported ID mapping


True broken references: 1381
References to quarantined parents: 7937
Formatting-only matches: 194695
Unique quarantined rows: 7118
Mapped source labels remaining: 0


,source_file,rejection_file,rows,reason_codes
0,orders.csv,date_rejects.csv,1605,order_date:INVALID_DATE | order_date:OUT_OF_AL...
1,payments.csv,date_rejects.csv,1067,payment_date:INVALID_DATE
2,returns.csv,date_rejects.csv,136,return_date:INVALID_DATE
3,sellers.csv,date_rejects.csv,4,join_date:INVALID_DATE
4,shipping.csv,date_rejects.csv,1942,delivered_date:INVALID_DATE | shipped_date:INV...
5,order_items.csv,invalid_order_items.csv,1588,MISSING_OR_UNPARSEABLE_QUANTITY | NEGATIVE_UNI...
6,order_items.csv,sentinel_order_items.csv,305,SUSPECT_SENTINEL_VALUE:unit_price
7,orders.csv,sentinel_orders.csv,145,SUSPECT_SENTINEL_VALUE:order_total
8,payments.csv,sentinel_payments.csv,162,SUSPECT_SENTINEL_VALUE:amount
9,shipping.csv,sentinel_shipping.csv,164,SUSPECT_SENTINEL_VALUE:shipping_fee


## Numeric and date review

Sentinel values and invalid order-item records were quarantined earlier in the workflow. The remaining negative values are reason-coded flags, not deletions. IQR extremes are summarized and retained because statistical distance alone does not establish an incorrect value. Ambiguous or invalid customer registration dates remain flagged; they are not guessed.

In [4]:
types = read_report("data_type_validation.csv")
outliers = read_report("final_outlier_audit.csv")
date_flags_path = PROJECT_ROOT / "data" / "rejected" / "date_field_flags.csv"
numeric_flags_path = PROJECT_ROOT / "data" / "rejected" / "numeric_anomaly_flags.csv"
print("Customer registration date flags:", len(pd.read_csv(date_flags_path)))
print("Current negative-value flags:", len(pd.read_csv(numeric_flags_path)))
display(types)
display(outliers)

Customer registration date flags: 1731
Current negative-value flags: 988


,file,column,target_type,rows,null_before,unparseable_non_null,null_after,dtype_written
0,customers.csv,birth_year,numeric,45000,4600,0,4600,Float64
1,customers.csv,registration_date,date (ISO text in CSV),45000,0,1731,0,ISO-8601 text plus flagged unresolved values
2,customers.csv,city_id,nullable integer,45000,2233,0,2233,Int64
3,customers.csv,birth_year,nullable integer,45000,4600,0,4600,Int64
4,locations.csv,latitude,numeric,650,33,0,33,Float64
5,locations.csv,longitude,numeric,650,31,0,31,Float64
6,locations.csv,city_id,nullable integer,650,0,0,0,Int64
7,orders.csv,order_total,numeric,158250,0,0,0,Int64
8,orders.csv,order_date,date (ISO text in CSV),158250,0,0,0,ISO-8601 text plus flagged unresolved values
9,orders.csv,ship_city_id,nullable integer,158250,6322,0,6322,Int64


,file,column,non_missing_values,negative_values_flagged,minimum,maximum,iqr_lower_bound,iqr_upper_bound,statistical_outlier_count,handling
0,customers.csv,birth_year,40400,0,1900.0000,2.030000e+03,1966.50000,2018.50000,236,Statistical extremes retained; no business evi...
1,locations.csv,latitude,617,0,23.3071,7.296160e+01,20.55745,39.77065,6,Statistical extremes retained; no business evi...
2,locations.csv,longitude,619,0,28.2568,7.900020e+01,62.54155,79.44155,8,Statistical extremes retained; no business evi...
3,order_items.csv,quantity,313250,0,1.0000,5.000000e+02,-0.50000,3.50000,20029,Statistical extremes retained; no business evi...
4,order_items.csv,unit_price,313250,0,0.0000,3.801600e+06,-2280.50000,6211.50000,25979,Statistical extremes retained; no business evi...
5,order_items.csv,discount_amount,281886,0,0.0000,1.630620e+05,-280.50000,467.50000,46003,Statistical extremes retained; no business evi...
6,orders.csv,order_total,158250,163,-78112.0000,3.261500e+05,-9012.00000,21036.00000,12184,Statistical extremes retained; no business evi...
7,payments.csv,amount,163586,656,-143168.0000,3.261500e+05,-8912.50000,20499.50000,12717,Statistical extremes retained; no business evi...
8,products.csv,list_price,3200,4,-3810.0000,9.999999e+06,-2300.00000,6340.00000,308,Statistical extremes retained; no business evi...
9,products.csv,cost_price,3030,0,82.0000,8.320500e+04,-1514.12500,4016.87500,347,Statistical extremes retained; no business evi...


## Staging handoff

All nine processed tables pass row reconciliation and have no remaining exact-row or primary-key duplicates. The files are suitable for unconstrained SQL Server landing/staging with nullable columns. Foreign-key-constrained loading remains blocked pending disposition of 1,381 truly broken references and 7,937 references to quarantined parents. Customer registration-date values also need an explicit parse policy before strict date typing.

No star schema, Airflow DAG, or Power BI dashboard is created by this milestone.